# Simple CNN testing

In [ ]:
import torch
import os
# os.environ["WANDB__SERVICE_WAIT"] = "500"
# os.environ["WANDB_DISABLE_SERVICE"] = "true"
os.environ["WANDB_MODE"] = "offline"
import wandb
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [ ]:
# Check for GPU
device = None
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else: 
    device = torch.device("cpu")

print(device)

In [ ]:
transformer = transforms.Compose([
    transforms.Resize((224, 224)), # transform to size = 224x224
    transforms.ToTensor(), # transforms into tensor
])

full_dataset = datasets.ImageFolder("../data/icosimal_img_class_03/train", transform=transformer)
splits = torch.load("../data/split/split_train_test_indices.pth")

train_dataset = torch.utils.data.Subset(full_dataset, splits['train_idx'])
test_dataset = torch.utils.data.Subset(full_dataset, splits['test_idx'])
val_dataset = datasets.ImageFolder("../data/icosimal_img_class_03/validate", transform=transformer)

In [ ]:
# check dataset loaded correctly
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

### Create Training loop for models

In [ ]:
def train_eval(model, optimizer, nepochs, batch_size, training_data, validation_data, device, entity='MSE_DeLearn_SPR26', project='MPW-CNN',run_name=None, use_wandb=True):
    """
    Train and evaluate a model.
    Logs train/validation loss and accuracy to Weights & Biases if use_wandb=True.
    """
    cost_hist = []
    cost_hist_test = []
    acc_hist = []
    acc_hist_test = []

    model = model.to(device) # <-- move model to device (GPU or CPU)
    cost_ce = torch.nn.CrossEntropyLoss().to(device)
    
    train_loader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(validation_data, batch_size=batch_size, shuffle=False)
    
    if use_wandb:
        wandb.init(
            entity=entity,
            project=project,
            name=run_name,
            settings=wandb.Settings(init_timeout=300),
            config={
                "epochs": nepochs,
                "batch_size": batch_size,
                "optimizer": optimizer.__class__.__name__,
                "loss": "CrossEntropyLoss",
                "device": str(device),
                "model": model.__class__.__name__
            }
        )
        wandb.watch(model, log="all", log_freq=100)

    for epoch in range(nepochs):
        model.train()
        size = len(train_loader.dataset)
        nbatches = len(train_loader)
        cost, acc = 0.0, 0.0
        for batch, (X, Y) in enumerate(train_loader):
            X,Y = X.to(device),Y.to(device)
            pred = model(X)
            loss = cost_ce(pred, Y)
            cost += loss.item()
            acc += (pred.argmax(dim=1) == Y).type(torch.float).sum().item()

            # gradient, parameter update
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
        cost /= nbatches
        acc /= size

        model.eval()
        size_test = len(val_loader.dataset)
        nbatches_test = len(val_loader)
        cost_test, acc_test = 0.0, 0.0     

        with torch.no_grad():
            for X, Y in val_loader:
                X,Y = X.to(device),Y.to(device)
                pred = model(X)
                cost_test += cost_ce(pred, Y).item()
                acc_test += (pred.argmax(dim=1) == Y).type(torch.float).sum().item()

        cost_test /= nbatches_test
        acc_test /= size_test

        print("Epoch %i: %f, %f, %f, %f"%(epoch, cost, acc, cost_test, acc_test))

        cost_hist.append(cost)
        cost_hist_test.append(cost_test)
        acc_hist.append(acc)
        acc_hist_test.append(acc_test)

        if use_wandb:
            wandb.log({
                "epoch": epoch + 1,
                "train_loss": cost,
                "train_accuracy": acc,
                "val_loss": cost_test,
                "val_accuracy": acc_test,
                "lr": optimizer.param_groups[0]['lr']
            })

    if use_wandb:
        wandb.finish()

    return cost_hist, cost_hist_test, acc_hist, acc_hist_test

### Creating simple CNN-model

In [ ]:
# creata simple model with one convolutional layer and two fully connected layers

class simple_model(nn.Module):
    
    def __init__(self, units=100):
        super(simple_model, self).__init__()
        self.seq = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # Conv with 32 filters, kernel size 3x3, padding 1
            nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Flatten(),
            nn.Linear(112*112*32,units),
            nn.ReLU(),
            nn.Linear(units,10) # output layer with 10 units for 10 classes
        )
        
    
    def forward(self, x):
        return self.seq(x)

In [ ]:
# create an model and its summary

model = simple_model(100).to(device)
from torchsummary import summary
summary(model, (3,224,224))

Initiate Training

In [ ]:
batch_size = 32
nepochs = 10
lr = 0.1
units = 100

model = simple_model(units)
optimizer = torch.optim.SGD(params=model.parameters(), lr = lr)
cost_train_sgd, cost_valid_sgd, acc_train_sgd, acc_valid_sgd = train_eval(model, optimizer, nepochs, batch_size, train_dataset, val_dataset, device, entity='MSE_DeLearn_SPR26', project='MPW-CNN', run_name='simple_model', use_wandb=True)


In [ ]:
import random

import wandb

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="MSE_DeLearn_SPR26",
    # Set the wandb project where this run will be logged.
    project="MPW-CNN",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 0.02,
        "architecture": "CNN",
        "dataset": "CIFAR-100",
        "epochs": 10,
    },
)

# Simulate training.
epochs = 10
offset = random.random() / 5
for epoch in range(2, epochs):
    acc = 1 - 2**-epoch - random.random() / epoch - offset
    loss = 2**-epoch + random.random() / epoch + offset

    # Log metrics to wandb.
    run.log({"acc": acc, "loss": loss})

# Finish the run and upload any remaining data.
run.finish()